# Dual OTel ingest (Arize AX + Unity Catalog)

This notebook demonstrates **split-stream tracing** for agents running outside Databricks-native auto-persist:

1. **Arize AX** — sub-second OTLP ingest for trace exploration, evaluation, and monitoring.
2. **Databricks governed storage** — the same spans exported via OTLP to Unity Catalog Delta tables for retention, governance, and SQL analytics.

**Demo workload:** OpenAI + LangChain tool-calling agent with OpenInference auto-instrumentation.

**Prerequisites:** Unity Catalog workspace, [OpenTelemetry on Databricks](https://docs.databricks.com/aws/en/mlflow3/genai/tracing/trace-unity-catalog) preview enabled, Arize space API key, OpenAI API key.

This notebook is **self-contained** — dependencies and dual-export helpers are defined inline (no separate `requirements.txt` or helper modules).

## 1. Install dependencies

In [0]:
%pip install \
  "mlflow[databricks]>=3.11.0" \
  "arize-otel>=0.8.0" \
  "openinference-instrumentation-langchain>=0.1.30" \
  "langchain>=1.0.0" \
  "langchain-core>=0.3.0" \
  "langchain-openai>=0.2.0" \
  "opentelemetry-exporter-otlp-proto-http>=1.27.0" \
  "openinference-semantic-conventions>=0.1.12" \
  --quiet
dbutils.library.restartPython()

## 2. Configuration

Run this section **after** the install cell restarts Python.

Set widgets below or map secrets to environment variables in your cluster policy.

| Variable | Description |
|----------|-------------|
| `catalog_name` / `schema_name` / `table_prefix` | UC trace location |
| `experiment_name` | MLflow experiment (optional UI for traces) |
| `sql_warehouse_id` | Warehouse for trace search / SQL |
| `arize_project_name` | Arize project for spans |
| Secrets: `ARIZE_SPACE_ID`, `ARIZE_API_KEY`, `DATABRICKS_HOST`, `DATABRICKS_TOKEN`, `OPENAI_API_KEY` | |

Access Secrets from Databricks Secrets and set them as Environment Variables
- Create a [Arize API key and Space ID](https://docs.arize.com/arize/reference/authentication-and-security/api-keys) for the items below.
- Use [Databricks Secrets](https://docs.databricks.com/aws/en/security/secrets/) for secure access of keys.

In [0]:
import os

dbutils.widgets.text("catalog_name", "main", "UC catalog")
dbutils.widgets.text("schema_name", "otel_traces", "UC schema")
dbutils.widgets.text("table_prefix", "partner_demo", "UC table prefix")
dbutils.widgets.text(
    "experiment_name",
    "/Shared/partner-dual-ingest-demo",
    "MLflow experiment",
)
dbutils.widgets.text("sql_warehouse_id", "default-warehouse-id", "SQL warehouse ID")
dbutils.widgets.text("arize_project_name", "databricks-dual-ingest-demo", "Arize project")
dbutils.widgets.text("openai_model", "gpt-4o-mini", "OpenAI model")
dbutils.widgets.text("demo_user_message", "What does our travel policy say about demo expenses?", "Demo prompt")

# Optional: load from a secret scope (uncomment and set scope name)
scope = "arize-databricks-partner"
os.environ["ARIZE_SPACE_ID"] = dbutils.secrets.get(scope=scope, key="arize-space-id")
os.environ["ARIZE_API_KEY"] = dbutils.secrets.get(scope=scope, key="arize-api-key")
os.environ["DATABRICKS_HOST"] = dbutils.secrets.get(scope=scope, key="databricks-host")
os.environ["DATABRICKS_TOKEN"] = dbutils.secrets.get(scope=scope, key="databricks-token")
os.environ["OPENAI_API_KEY"] = dbutils.secrets.get(scope=scope, key="openai-api-key")

catalog_name = dbutils.widgets.get("catalog_name").strip()
schema_name = dbutils.widgets.get("schema_name").strip()
table_prefix = dbutils.widgets.get("table_prefix").strip()
experiment_name = dbutils.widgets.get("experiment_name").strip()
sql_warehouse_id = dbutils.widgets.get("sql_warehouse_id").strip()
arize_project_name = dbutils.widgets.get("arize_project_name").strip()
openai_model = dbutils.widgets.get("openai_model").strip()
demo_user_message = dbutils.widgets.get("demo_user_message").strip()

if not sql_warehouse_id:
    raise ValueError("Set the sql_warehouse_id widget to a warehouse you can use.")

os.environ["MLFLOW_TRACING_SQL_WAREHOUSE_ID"] = sql_warehouse_id

# Default workspace URL when running on Databricks (override with DATABRICKS_HOST if needed)
if not os.environ.get("DATABRICKS_HOST"):
    try:
        os.environ["DATABRICKS_HOST"] = (
            dbutils.notebook.entry_point.getDbutils()
            .notebook()
            .getContext()
            .apiUrl()
            .get()
            .rstrip("/")
        )
    except Exception:
        pass

## 3. Bind MLflow experiment to Unity Catalog (public preview)

Creates `{prefix}_otel_spans` (and related tables) and links the experiment for optional trace UI browsing.

In [0]:
import mlflow
from mlflow.entities.trace_location import UnityCatalog

mlflow.set_tracking_uri("databricks")

experiment = mlflow.set_experiment(
    experiment_name=experiment_name,
    trace_location=UnityCatalog(
        catalog_name=catalog_name,
        schema_name=schema_name,
        table_prefix=table_prefix,
    ),
)

uc_spans_table = experiment.trace_location.full_otel_spans_table_name
print(f"Experiment ID: {experiment.experiment_id}")
print(f"UC spans table: {uc_spans_table}")

## 4. Configure dual OTLP export

One `TracerProvider`, two span processors — **Arize (gRPC)** and **Databricks (OTLP HTTP)**.

In [0]:
import os
from typing import Optional

from opentelemetry import trace
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor

from arize.otel import BatchSpanProcessor as ArizeBatchSpanProcessor
from arize.otel import Endpoint
from arize.otel import GRPCSpanExporter as ArizeGRPCSpanExporter
from arize.otel import HTTPSpanExporter as ArizeHTTPSpanExporter
from arize.otel import Transport
from openinference.semconv.resource import ResourceAttributes


def _arize_endpoint():
    raw = (os.environ.get("ARIZE_COLLECTOR_ENDPOINT") or "").strip()
    if "eu-west" in raw:
        return Endpoint.ARIZE_EUROPE
    if raw.startswith("http://") or raw.startswith("https://"):
        return raw.rstrip("/")
    return Endpoint.ARIZE


def _arize_transport():
    mode = (os.environ.get("ARIZE_TRANSPORT") or "grpc").strip().lower()
    return Transport.HTTP if mode == "http" else Transport.GRPC


def _require(name: str, value: Optional[str]) -> str:
    if not value or not str(value).strip():
        raise ValueError(f"Missing required configuration: {name}")
    return str(value).strip()


def configure_dual_export(
    *,
    project_name: str,
    uc_spans_table: str,
    service_name: str = "langchain-external-agent-demo",
) -> TracerProvider:
    project_name = _require("project_name", project_name)
    space_id = _require(
        "arize_space_id",
        os.environ.get("ARIZE_SPACE_ID") or os.environ.get("ARIZE_SPACE"),
    )
    api_key = _require("arize_api_key", os.environ.get("ARIZE_API_KEY"))
    host = _require("databricks_host", os.environ.get("DATABRICKS_HOST")).rstrip("/")
    token = _require("databricks_token", os.environ.get("DATABRICKS_TOKEN"))

    resource = Resource.create(
        {
            ResourceAttributes.PROJECT_NAME: project_name,
            "openinference.project.name": project_name,
            "service.name": service_name,
            "model_id": project_name,
        }
    )
    provider = TracerProvider(resource=resource)

    endpoint = _arize_endpoint()
    transport = _arize_transport()
    if transport == Transport.GRPC:
        arize_exporter = ArizeGRPCSpanExporter(
            space_id=space_id, api_key=api_key, endpoint=endpoint
        )
    else:
        arize_exporter = ArizeHTTPSpanExporter(
            space_id=space_id, api_key=api_key, endpoint=endpoint
        )

    provider.add_span_processor(ArizeBatchSpanProcessor(span_exporter=arize_exporter))
    print(
        f"Arize export: transport={transport.value}, endpoint={endpoint}, project={project_name}"
    )

    dbx_exporter = OTLPSpanExporter(
        endpoint=f"{host}/api/2.0/otel/v1/traces",
        headers={
            "content-type": "application/x-protobuf",
            "X-Databricks-UC-Table-Name": uc_spans_table,
            "Authorization": f"Bearer {token}",
        },
    )
    provider.add_span_processor(BatchSpanProcessor(dbx_exporter))
    trace.set_tracer_provider(provider)
    return provider


def shutdown_tracer(provider: TracerProvider) -> None:
    provider.force_flush()
    provider.shutdown()


tracer_provider = configure_dual_export(
    project_name=arize_project_name,
    uc_spans_table=uc_spans_table,
    service_name="langchain-external-agent-demo",
)

## 5. Instrument LangChain and run the demo agent

In [0]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from openinference.instrumentation.langchain import LangChainInstrumentor

LangChainInstrumentor().instrument(tracer_provider=tracer_provider)

@tool
def lookup_policy(topic: str) -> str:
    """Return a short internal policy snippet for the given topic."""
    policies = {
        "travel": "Demo travel expenses under $500 are pre-approved for partner workshops.",
        "security": "All agent outputs must stay within approved data boundaries.",
    }
    return policies.get(topic.lower(), f"No policy on file for '{topic}'.")

llm = ChatOpenAI(model=openai_model, temperature=0)
tools = [lookup_policy]

agent = create_agent(
    llm,
    tools=tools,
    system_prompt=(
        "You are an enterprise assistant. Use tools when you need policy details. Be concise."
    ),
)

print("Running demo agent...")
result = agent.invoke({"messages": [{"role": "user", "content": demo_user_message}]})

messages = result.get("messages", [])
print("\n--- Agent response ---")
print(messages[-1].content if messages else result)

## 6. Flush spans

Ensures OTLP batches are delivered before verification (required for short notebook runs).

In [0]:
shutdown_tracer(tracer_provider)
print("Tracer shut down and spans flushed.")

## 7. Verify Unity Catalog ingest (Databricks SQL)

Spans should appear in `{catalog}.{schema}.{prefix}_otel_spans` within a short delay after export.

In [0]:
import time
time.sleep(15)

display(
    spark.sql(f"""
        SELECT
          trace_id,
          span_id,
          name,
          kind,
          start_time_unix_nano,
          end_time_unix_nano,
          (end_time_unix_nano - start_time_unix_nano) / 1e6 AS duration_ms
        FROM {uc_spans_table}
        ORDER BY start_time_unix_nano DESC
        LIMIT 10
    """)
)

## 8. Verify Arize AX

Open your Arize project (widget: `arize_project_name`) in the AX UI. You should see the same agent run with LLM and tool spans.

If traces are missing:
- Confirm `ARIZE_SPACE_ID` and `ARIZE_API_KEY`
- Confirm `arize_project_name` is set (required resource attribute)
- Re-run after checking cluster env vars

In [0]:
print(
    f"Check Arize AX → project '{arize_project_name}' for traces from service "
    "'langchain-external-agent-demo'."
)

## 9. Optional: MLflow Experiment UI

In the workspace **Experiments** page, open `{experiment_name}` → **Traces** tab (select your SQL warehouse). This is optional; SQL on the UC spans table is the primary governed-store proof.

In [0]:
print(f"MLflow experiment: {experiment_name}")
print(f"Experiment ID: {experiment.experiment_id}")